# 🚀 개선된 VQA 학습/추론 파이프라인

Qwen2.5-VL-3B-Instruct + LoRA Fine-tuning

### 주요 개선사항
| 항목 | 기존 | 개선 |
|------|------|------|
| 학습 데이터 | 200개 샘플 | **전체 데이터** |
| 에폭 수 | 1 | **3 + Early Stopping** |
| LoRA rank / alpha | r=8, α=16 | **r=32, α=64** |
| 이미지 해상도 | 384×384 | **512×512** |
| Grad Accum (eff. batch) | 4 | **8** |
| LR 스케줄러 | Linear | **Cosine + Warmup 10%** |
| Loss 계산 | 전체 토큰 | **답변 토큰만 (Label Masking)** |
| Gradient Clipping | 없음 | **max_norm=1.0** |
| 추론 | 1개씩 | **배치 추론 (4)** |
| Compute dtype | float16 | **bfloat16** |
| Weight Decay | 없음 | **0.01** |
| DataLoader workers | 0 | **2 + pin_memory** |

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

In [ ]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

In [ ]:
import os, re, math, random, gc
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===== 하이퍼파라미터 =====
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_SIZE = 512           # ★ 384 → 512
MAX_NEW_TOKENS = 4
SEED = 42
NUM_EPOCHS = 3             # ★ 1 → 3
BATCH_SIZE = 1
GRAD_ACCUM = 8             # ★ 4 → 8 (effective batch = 8)
LEARNING_RATE = 2e-4       # ★ 1e-4 → 2e-4
WARMUP_RATIO = 0.1         # ★ 0.03 → 0.1
INFERENCE_BATCH = 4        # ★ 배치 추론
PATIENCE = 2               # ★ Early stopping

random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); np.random.seed(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print(f"학습 데이터: {len(train_df)}개, 테스트 데이터: {len(test_df)}개")

# ★ 전체 데이터 사용 (200개 샘플링 제거)


In [ ]:
# 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # ★ float16 → bfloat16
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 256,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# ★ LoRA 강화: r=32, alpha=64
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ★ 강화된 시스템 프롬프트
SYSTEM_INSTRUCT = (
    "You are an expert visual question answering system. "
    "Look at the image carefully, analyze the question, and select the single best answer. "
    "You must respond with ONLY one lowercase letter: a, b, c, or d. "
    "Do not include any explanation, punctuation, or extra text."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"Question: {question}\n\n"
        f"Options:\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "Based on the image, the correct answer is:"
    )


In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images, prompt_texts = [], [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            full_text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(full_text)
            images.append(img)

            if self.train:
                # ★ 프롬프트만 (답변 제외) → label masking용
                prompt_msgs = messages[:-1]
                prompt_text = self.processor.apply_chat_template(
                    prompt_msgs, tokenize=False, add_generation_prompt=True
                )
                prompt_texts.append(prompt_text)

        enc = self.processor(text=texts, images=images, padding=True, return_tensors="pt")

        if self.train:
            labels = enc["input_ids"].clone()
            # ★ 프롬프트 부분을 -100으로 마스킹 → 답변 토큰만 loss 계산
            for idx, pt in enumerate(prompt_texts):
                prompt_enc = self.processor.tokenizer(pt, return_tensors="pt", add_special_tokens=False)
                prompt_len = prompt_enc["input_ids"].shape[1]
                labels[idx, :prompt_len] = -100
            # padding도 -100
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            enc["labels"] = labels

        return enc


In [ ]:
# 데이터 셔플 후 분할
train_df_shuffled = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split = int(len(train_df_shuffled) * 0.9)
train_subset = train_df_shuffled.iloc[:split]
valid_subset = train_df_shuffled.iloc[split:]
print(f"학습: {len(train_subset)}개, 검증: {len(valid_subset)}개")

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=2, pin_memory=True, prefetch_factor=2,   # ★ 병렬 로딩
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=2, pin_memory=True,
)


In [ ]:
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)
scaler = torch.amp.GradScaler("cuda", enabled=True)

best_val_loss = float("inf")
patience_counter = 0
SAVE_DIR = "qwen2_5_vl_3b_lora_best"

print(f"총 학습 스텝: {num_training_steps}, 워밍업: {num_warmup_steps}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

global_step = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    running = 0.0
    epoch_loss = 0.0
    epoch_steps = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ★ Gradient Clipping
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            epoch_loss += avg_loss
            epoch_steps += 1
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            running = 0.0

    # 남은 gradient flush
    if step % GRAD_ACCUM != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

    # Validation
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]", unit="batch"):
            vb = {k: v.to(device, non_blocking=True) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1

    avg_val = val_loss / max(val_steps, 1)
    avg_train = epoch_loss / max(epoch_steps, 1)
    print(f"
[Epoch {epoch+1}] train loss: {avg_train:.4f} | valid loss: {avg_val:.4f}")

    # ★ Early Stopping
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_counter = 0
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  → Best model saved! (val_loss: {best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  → No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("  → Early stopping triggered!")
            break

    gc.collect()
    torch.cuda.empty_cache()

# ★ Best 모델 로드
print(f"
Best val loss: {best_val_loss:.4f}")
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, SAVE_DIR)
model = model.to(device)
print("Best model loaded!")


In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines: return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]: return last

    for pat in [r"\b([abcd])\b", r"\(([abcd])\)", r"answer\s*(?:is|:)\s*([abcd])"]:
        m = re.search(pat, text)
        if m: return m.group(1)
    return "a"


def batch_inference(model, processor, test_df, batch_size=4):
    model.eval()
    preds = []
    for start in tqdm(range(0, len(test_df), batch_size), desc="Inference", unit="batch"):
        end = min(start + batch_size, len(test_df))
        batch_rows = test_df.iloc[start:end]
        texts, images = [], []

        for _, row in batch_rows.iterrows():
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])
            messages = [
                {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
                {"role":"user","content":[
                    {"type":"image","image":img},
                    {"type":"text","text":user_text}
                ]}
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            texts.append(text)
            images.append(img)

        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(device)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            out_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, eos_token_id=processor.tokenizer.eos_token_id,
            )
        for txt in processor.batch_decode(out_ids, skip_special_tokens=True):
            preds.append(extract_choice(txt))
        del inputs, out_ids
        torch.cuda.empty_cache()
    return preds


preds = batch_inference(model, processor, test_df, batch_size=INFERENCE_BATCH)

os.makedirs("content", exist_ok=True)
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("content/submission.csv", index=False)
print(f"Saved content/submission.csv ({len(submission)}개)")
print(submission["answer"].value_counts())
